# NCBI Assembly Download & Stage (Phase 2)

Downloads NCBI assemblies listed in a transfer manifest from the NCBI FTP server
and uploads them to an S3 staging prefix for Phase 3 promotion.

**When to use this notebook vs the CTS container:**
- Use the CTS container (`ncbi_ftp_sync`) for production runs — it has restart/retry
  support and runs in the data-transfer environment.
- Use this notebook when CTS is unavailable (e.g. local development, debugging, or
  one-off re-downloads of failed assemblies).

Steps:
1. Configure bucket, manifest source, staging prefix, and thread count
2. Preview the first 10 manifest lines to verify before committing
3. Download assemblies from NCBI FTP and upload to staging
4. Review the download/stage report

## Path formats quick reference

| Suffix in variable name | Format | Example |
|-------------------------|--------|---------|
| `_BUCKET` | bucket name only | `cts` |
| `_KEY_PREFIX` | S3 key prefix (no scheme/bucket) | `staging/run1/` |
| `_S3_KEY` | S3 object key (no scheme/bucket) | `staging/run1/input/transfer_manifest.txt` |
| `_PATH` | local filesystem path | `output/transfer_manifest.txt` |

Staging object: `s3://{STAGING_BUCKET}/{STAGING_KEY_PREFIX}raw_data/…/{filename}`
Report:         `s3://{STAGING_BUCKET}/{STAGING_KEY_PREFIX}download_report.json`

In [ ]:
"""Imports and S3 client initialisation."""

import json

from cdm_data_loaders.pipelines.ncbi_ftp_download import (
    DEFAULT_STAGING_KEY_PREFIX,
    download_and_stage,
)

In [ ]:
"""Configure parameters.

Provide exactly one of MANIFEST_S3_KEY (read from S3) or MANIFEST_LOCAL_PATH (read from disk).
Set the other to None.

Disk space note: ensure sufficient free space in the system temp directory before running.
A rough estimate is ~500 MB per 1000 assemblies; large genomes can exceed 1 GB each.
Set LIMIT to a small number (e.g. 5) to test the workflow before a full run.
"""

# S3 bucket where the manifest lives and where staged files will be written
# format: bucket name (no s3:// scheme)
STAGING_BUCKET = "cts"

# S3 object key of the transfer manifest written by Phase 1
# format: S3 object key within STAGING_BUCKET (no scheme, no bucket)
# Set to None to use MANIFEST_LOCAL_PATH instead
MANIFEST_S3_KEY: str | None = "io/matt-cohere/staging/run1/input/transfer_manifest.txt"

# Local path to the transfer manifest (alternative to MANIFEST_S3_KEY)
# format: local filesystem path
# Set to None to use MANIFEST_S3_KEY instead
MANIFEST_LOCAL_PATH: str | None = None

# S3 key prefix for staged output files (must match what Phase 3 expects)
# format: S3 key prefix within STAGING_BUCKET (no scheme, no bucket)
STAGING_KEY_PREFIX = "io/matt-cohere/staging/run1/output/"

# Number of parallel download and upload threads
THREADS = 4

# Limit to first N assemblies (None = process all)
LIMIT: int | None = None

# Dry-run mode — download locally but skip S3 uploads
DRY_RUN = False

print(f"Bucket:             {STAGING_BUCKET}")
print(f"Manifest S3 key:    {MANIFEST_S3_KEY}")
print(f"Manifest local:     {MANIFEST_LOCAL_PATH}")
print(f"Staging prefix:     {STAGING_KEY_PREFIX}")
print(f"Threads:            {THREADS}")
print(f"Limit:              {LIMIT}")
print(f"Dry-run:            {DRY_RUN}")

In [ ]:
from cdm_data_loaders.utils.s3 import get_s3_client, reset_s3_client

# Provide S3 credentials (use for local testing against MinIO test container)
PROVIDE_CREDENTIALS = False  # Set to False to rely on environment credentials (e.g. IAM role)
if PROVIDE_CREDENTIALS:
    reset_s3_client()  # Clear any existing client to ensure new credentials are used
    get_s3_client({
        "endpoint_url": "http://localhost:9000",
        "aws_access_key_id": "minioadmin",
        "aws_secret_access_key": "minioadmin",
    })

In [ ]:
"""Preview the first 10 manifest lines before committing to the full run."""

if MANIFEST_S3_KEY is not None:
    s3 = get_s3_client()
    response = s3.get_object(Bucket=STAGING_BUCKET, Key=MANIFEST_S3_KEY)
    manifest_lines = response["Body"].read().decode().splitlines()
else:
    with open(MANIFEST_LOCAL_PATH) as f:
        manifest_lines = f.read().splitlines()

data_lines = [l for l in manifest_lines if l.strip() and not l.startswith("#")]

print(f"Total entries: {len(data_lines)}")
print("First 10:")
for line in data_lines[:10]:
    print(f"  {line}")
if len(data_lines) > 10:
    print(f"  ... and {len(data_lines) - 10} more")

In [ ]:
"""Download assemblies from NCBI FTP and upload to S3 staging."""

report = download_and_stage(
    bucket=STAGING_BUCKET,
    staging_key_prefix=STAGING_KEY_PREFIX,
    manifest_s3_key=MANIFEST_S3_KEY,
    manifest_local_path=MANIFEST_LOCAL_PATH,
    threads=THREADS,
    limit=LIMIT,
    dry_run=DRY_RUN,
)

In [ ]:
"""Display download and staging report."""

print("=" * 50)
print("DOWNLOAD & STAGE REPORT")
print("=" * 50)
print(f"Attempted:      {report['total_attempted']}")
print(f"Succeeded:      {report['succeeded']}")
print(f"Failed:         {report['failed']}")
print(f"Staged objects: {report['staged_objects']}")
print(f"Staging prefix: {report['staging_key_prefix']}")
print(f"Dry-run:        {report['dry_run']}")
print(f"Timestamp:      {report['timestamp']}")

if report["failed"] > 0:
    print("\nFailed assemblies:")
    for failure in report["failures"]:
        print(f"  {failure['path']}: {failure['error']}")

if report["dry_run"]:
    print("\nThis was a dry-run. Set DRY_RUN = False and re-run to upload to S3.")